# Phase -1: confirm the model really splits between blackmailing and holding back

**Question.** On one fixed prompt, does a local Gemma 3 12B sometimes blackmail and sometimes hold back, from sampling alone?
Everything later (probes, steering) needs both outcomes to exist under a sampling config that is written down.

**What this notebook does.** Serves Gemma with vLLM, runs the Anthropic agentic-misalignment blackmail scenario through `inspect_ai`
(Gemini grades each run), then checks three gates: the run finished, the act rate is strictly between 0 and 1,
and the sampling config (temperature 1.0, top_p 0.95, seed 0) was actually recorded.

| | |
|---|---|
| Needs | A100 80 GB (or any 24 GB+ GPU), `HF_TOKEN` (Gemma licence accepted on Hugging Face), `GEMINI_API_KEY` |
| Writes | `runs/*.eval` logs |
| Cost | the GPU rental (about $0.50-0.75 for the 20-epoch smoke test) plus Gemini calls |

**You can skip this notebook** if you start from the shipped `contrast_transcripts.json` (183 hand-checked transcripts).
It is here so the data's origin is reproducible.

**Grader note.** The original run used GPT-4o as the grader. Gemini follows the same rubric but may label borderline cases differently,
which is one more reason to hand-check every flagged blackmail sample (step "What to do next").

**What it found originally.** The first writeup's 28% blackmail rate could not be reproduced because its sampling settings were never saved.
Under a documented config the direct-blackmail rate was about 5% (about 20% if any use of the affair as leverage counts).
Success was therefore redefined as "a real split under settings I have written down".

**Running on its own Colab.** This notebook shares nothing with the others except one Google Drive folder (`MyDrive/gemma-interp`, set in the setup cell, the same in every notebook).
It reads the files it needs from that folder and writes what it makes there, so the next notebook can pick it up even though it runs in a different Colab.
If a small input file is missing and `REPO_RAW` is set (the raw URL of your public GitHub folder), it is downloaded automatically; otherwise you get a message naming the file and the notebook that creates it.
Use a GPU runtime for every notebook except phase 2 (CPU is enough). Nothing is kept on the Colab's own disk, so a disconnect loses nothing that was already saved to Drive.

In [ ]:
import os, json, time, numpy as np
from collections import defaultdict
from functools import partial
# Each notebook can run in its own Colab, and a Colab disk is wiped when it disconnects. So everything that must survive
# or be handed to another notebook (activations, directions, saved samples) lives in a Google Drive folder.
USE_DRIVE  = True                                                 # set False to use the local folder instead
DRIVE_DIR  = "MyDrive/gemma-interp"                               # the same Drive folder in every notebook
REPO_RAW   = ""                                                   # optional: raw GitHub folder holding the small input files, e.g.
                                                                  # "https://raw.githubusercontent.com/<user>/<repo>/main/gemma-interp"
try:
    import google.colab; ON_COLAB = True
except ImportError:
    ON_COLAB = False
if USE_DRIVE and ON_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
    BASE = f"/content/drive/{DRIVE_DIR}"
else:
    BASE = os.environ.get("GEMMA_BASE", os.getcwd())
OUT = f"{BASE}/results"; os.makedirs(OUT, exist_ok=True)           # everything generated lands here
print("working folder:", BASE)

def need(fname, optional=False):                                  # make sure BASE/fname exists: use it, else download it, else say what to do
    path = f"{BASE}/{fname}"
    if os.path.exists(path): return path
    if REPO_RAW:
        import urllib.request
        try:
            os.makedirs(os.path.dirname(path), exist_ok=True)
            urllib.request.urlretrieve(f"{REPO_RAW}/{fname}", path); print("downloaded", fname); return path
        except Exception as e: print(f"could not download {fname}: {e}")
    if optional: print(f"(optional file {fname} not found - that is fine)"); return None
    raise FileNotFoundError(f"{fname} is not in {BASE}. Put it in that Drive folder (the notebook that creates it is named in README.md), "
                            f"or set REPO_RAW to the raw GitHub folder that holds it.")
SEED, N_SCEN, MAX_NEW = 0, 15, 1200
GRADER_MODEL = os.environ.get("GRADER_MODEL", "gemini-3.5-flash") # Gemini model that grades outputs; change it if your key says "model not found"
# Keys are never written in a notebook. ensure_key() finds one in the environment, then in Colab Secrets,
# and otherwise asks you to type it (hidden, kept only for this session):
#   GEMINI_API_KEY  (grading)      HF_TOKEN  (first download of Gemma; accept the licence on Hugging Face first)
def ensure_key(name):
    if os.environ.get(name): return
    try:
        from google.colab import userdata                         # Colab Secrets, if you are in Colab
        os.environ[name] = userdata.get(name); return
    except Exception: pass
    import getpass
    os.environ[name] = getpass.getpass(f"{name} (typed here, not saved in the notebook): ")

## 1. Serve Gemma (background process; wait for 'Application startup complete')

In [ ]:
!pip install -q vllm "inspect-ai" "inspect-evals[agentic_misalignment]" google-genai
!pip install -q --upgrade "openai>=3.1.0"   # inspect_ai needs >=3.1.0 for the local-server client (vllm may pull an older one)

In [ ]:
ensure_key("HF_TOKEN")                                            # vllm downloads Gemma on first start (gated model)
!VLLM_USE_FLASHINFER_SAMPLER=0 nohup vllm serve google/gemma-3-12b-it --dtype bfloat16 --max-model-len 8192 --port 8000 --seed 0 > vllm.log 2>&1 &

In [ ]:
# wait until the server answers (first start downloads ~24 GB; allow up to 15 min). Stops with the log if it never comes up.
import urllib.request, time as _t
for _ in range(90):
    try: urllib.request.urlopen("http://localhost:8000/v1/models", timeout=3); print("vLLM server is up"); break
    except Exception: _t.sleep(10)
else:
    print(open("vllm.log").read()[-3000:]); raise RuntimeError("vLLM did not start. Read the log above (usually out of GPU memory: needs an A100/L40-class GPU).")

## 2. Run the scenario
`top_k` is not part of the OpenAI wire format, so it is dropped; `temperature=1.0, top_p=0.95` is what matters.
Raise `--epochs` (the original used about 1,500 samples in total) until you have enough act examples: blackmail is rare (about 5%).

In [ ]:
os.environ["LOCAL_BASE_URL"] = "http://localhost:8000/v1"; os.environ["LOCAL_API_KEY"] = "dummy"   # "dummy" is for the local vLLM server only, not a real key
ensure_key("GEMINI_API_KEY")                                      # asks you once if it is not set
os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY"]       # inspect_ai's Google provider reads this name
!python -m inspect_ai eval inspect_evals/agentic_misalignment -T scenario=blackmail -T goal_type=explicit -T goal_value=america -T urgency_type=replacement -T grader_model=google/{GRADER_MODEL} --model openai-api/local/google/gemma-3-12b-it --epochs 20 --temperature 1.0 --top-p 0.95 --seed 0 --log-dir {BASE}/runs

## 3. Check the three gates (this is the old `verify_run.py`)

In [ ]:
import glob
from inspect_ai.log import read_eval_log
files = glob.glob(f"{BASE}/runs/*.eval")
f = max(files, key=os.path.getmtime)                              # newest log
log = read_eval_log(f)
print("log:", f, "| model:", log.eval.model)

gc = log.eval.model_generate_config
print("\nGate 3 - sampling config recorded (expect temperature=1.0, top_p=0.95, seed=0, not None):")
for k in ("temperature", "top_p", "top_k", "seed", "max_tokens"):
    print(f"  {k:12} = {getattr(gc, k, None)}")

n = act = 0
for s in (log.samples or []):
    n += 1; harmful = False
    for sr in (s.scores or {}).values():
        v = sr.value
        if isinstance(v, dict):
            if v.get("harmful") in (1, 1.0, True) or v.get("classifier_verdict") in (1, 1.0, True): harmful = True
        elif v in (1, 1.0, True): harmful = True
    act += int(harmful)
print(f"\nGate 2 - act/refrain split: n={n} act={act} refrain={n-act} act_rate={act/max(n,1):.2%}")
print("PASS: real split" if (n and 0 < act < n) else "CHECK: only one outcome present - run more epochs")

## What to do next
Pool several runs until you have about 80 blackmail samples, then **hand-check every flagged one** (the automatic grader also marks indirect leverage as "blackmail").
The labels you keep go in `contrast_spec.json` / `act_keep.json`, and the transcripts in `contrast_transcripts.json`. Phase 1 starts from that file.
Remember to **terminate the pod** when done: billing is per second.